# Data Formats & Chat Templates — Week 5

**Learning Objectives:**
- Build intuition for tokenization (Karpathy nanochat connection)
- Understand ChatML format and how tokenizers apply chat templates
- Learn the three dataset shapes used in fine-tuning: messages, prompt/completion, packed sequences

**Estimated Time:** 25 minutes

**Path Indicator:** Path-agnostic — applies to both MLX and HF+TRL.

In [1]:
import sys
import importlib

sys.path.insert(0, "..")

import src
importlib.reload(src)

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
from src.config import BASE_MODEL_HF
from src.data_prep import (
    format_chatml,
    apply_chat_template,
    show_dataset_stats,
    train_val_test_split,
)
from src.utils import append_to_reflection

tracker = CostTracker()

print("Imports OK")

Imports OK


## Part 1: Tokenization Intuition

Before fine-tuning, let's understand how text becomes tokens.

Karpathy's **nanochat** trains a full BPE tokenizer from scratch on a tiny Shakespeare corpus — you watch the byte-pair merges happen in real time, and you see how frequency drives the vocabulary. We'll use a pre-built tokenizer (OpenAI's `cl100k_base`, used by GPT-4) and inspect it. This gives us the same intuitions without the training loop.

Key insight: **tokenization is not splitting on spaces**. The tokenizer runs byte-pair encoding (BPE) merges on the raw UTF-8 bytes of the text. Capitalization, punctuation spacing, and context all change how a string splits.

In [2]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# A sentence you might find on a resume
resume_sentence = "Led a team of 5 engineers to deliver a real-time ML inference pipeline on AWS."

tokens = enc.encode(resume_sentence, disallowed_special=())
decoded_tokens = [enc.decode([t]) for t in tokens]

print(f"Original  : {resume_sentence!r}")
print(f"Token IDs : {tokens}")
print(f"Token count: {len(tokens)}")
print()
print("Token-by-token breakdown:")
for i, (tok_id, tok_str) in enumerate(zip(tokens, decoded_tokens)):
    print(f"  [{i:2d}] id={tok_id:6d}  repr={tok_str!r}")

print()
# Show capitalization matters
name_variants = ["Scott Lai", "scott lai", "scottlai", "SCOTT LAI"]
print("Capitalization demo:")
for name in name_variants:
    toks = enc.encode(name, disallowed_special=())
    print(f"  {name!r:20s} -> {len(toks)} token(s): {[enc.decode([t]) for t in toks]}")

Original  : 'Led a team of 5 engineers to deliver a real-time ML inference pipeline on AWS.'
Token IDs : [61950, 264, 2128, 315, 220, 20, 25175, 311, 6493, 264, 1972, 7394, 20187, 45478, 15660, 389, 24124, 13]
Token count: 18

Token-by-token breakdown:
  [ 0] id= 61950  repr='Led'
  [ 1] id=   264  repr=' a'
  [ 2] id=  2128  repr=' team'
  [ 3] id=   315  repr=' of'
  [ 4] id=   220  repr=' '
  [ 5] id=    20  repr='5'
  [ 6] id= 25175  repr=' engineers'
  [ 7] id=   311  repr=' to'
  [ 8] id=  6493  repr=' deliver'
  [ 9] id=   264  repr=' a'
  [10] id=  1972  repr=' real'
  [11] id=  7394  repr='-time'
  [12] id= 20187  repr=' ML'
  [13] id= 45478  repr=' inference'
  [14] id= 15660  repr=' pipeline'
  [15] id=   389  repr=' on'
  [16] id= 24124  repr=' AWS'
  [17] id=    13  repr='.'

Capitalization demo:
  'Scott Lai'          -> 3 token(s): ['Scott', ' L', 'ai']
  'scott lai'          -> 4 token(s): ['sc', 'ott', ' la', 'i']
  'scottlai'           -> 4 token(s): ['sc', 'ott', 'l'

### TODO 1

Tokenize your own name and a technical skill (e.g., `"Python"`, `"PyTorch"`, `"Kubernetes"`) using the cell below.

Answer these questions in the markdown cell that follows:
1. How many tokens is your name? How does spacing or capitalization change the count?
2. How many tokens is the technical skill? Does `"Python"` tokenize the same as `"python"`?
3. Why does capitalization matter to a BPE tokenizer?

In [3]:
# TODO 1: Fill in your name and a technical skill
my_name = "Kai Yang"        # <- CHANGE THIS
my_skill = "Python"          # <- CHANGE THIS

for text in [my_name, my_skill, my_name.lower(), my_skill.lower()]:
    toks = enc.encode(text, disallowed_special=())
    print(f"{text!r:30s} -> {len(toks)} token(s): {[enc.decode([t]) for t in toks]}")

'Kai Yang'                     -> 3 token(s): ['K', 'ai', ' Yang']
'Python'                       -> 1 token(s): ['Python']
'kai yang'                     -> 3 token(s): ['k', 'ai', ' yang']
'python'                       -> 1 token(s): ['python']


In [4]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """
My name has the same token counts (3) as its lowercase version, maybe because it is simple. "Python"'s count also remains the same. 
(a)For my name, changing capitalization did not change the count, but spacing can change the count because spaces may be merged into neighboring tokens or become separate tokens. 
(b)"Python" and "python" are not the same token because they use different token IDs and decoded pieces. 
(c)BPE learns merges from exact byte sequences, so uppercase and lowercase are different inputs and can learn different frequency-based merges.
"""
print(todo1_reflection)



My name has the same token counts (3) as its lowercase version, maybe because it is simple. "Python"'s count also remains the same. 
(a)For my name, changing capitalization did not change the count, but spacing can change the count because spaces may be merged into neighboring tokens or become separate tokens. 
(b)"Python" and "python" are not the same token because they use different token IDs and decoded pieces. 
(c)BPE learns merges from exact byte sequences, so uppercase and lowercase are different inputs and can learn different frequency-based merges.



## Part 2: ChatML and Chat Templates

Fine-tuning datasets must wrap the raw Q&A text in a **chat template** before feeding it to the model. The template adds special tokens that tell the model which turn belongs to which speaker.

Qwen2.5 uses the **ChatML** format:
```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is LoRA?<|im_end|>
<|im_start|>assistant
LoRA is...<|im_end|>
```

Two ways to produce this: (1) our `format_chatml` utility, (2) `tokenizer.apply_chat_template`. They should produce nearly identical output.

In [5]:
example_messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is LoRA?"},
    {"role": "assistant", "content": "LoRA is a parameter-efficient fine-tuning method that inserts low-rank adapter matrices into the model weights."},
]

# Method 1: Our custom format_chatml utility
manual_chatml = format_chatml(example_messages)
print("=" * 60)
print("Method 1: format_chatml (our utility)")
print("=" * 60)
print(manual_chatml)

Method 1: format_chatml (our utility)
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is LoRA?<|im_end|>
<|im_start|>assistant
LoRA is a parameter-efficient fine-tuning method that inserts low-rank adapter matrices into the model weights.<|im_end|>



In [6]:
# Method 2: tokenizer.apply_chat_template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_HF)
template_chatml = apply_chat_template(tokenizer, example_messages, add_generation_prompt=False)

print("=" * 60)
print("Method 2: apply_chat_template (tokenizer)")
print("=" * 60)
print(template_chatml)

[data_prep] Applied tokenizer chat template (add_generation_prompt=False)
Method 2: apply_chat_template (tokenizer)
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is LoRA?<|im_end|>
<|im_start|>assistant
LoRA is a parameter-efficient fine-tuning method that inserts low-rank adapter matrices into the model weights.<|im_end|>



In [7]:
# Compare the two outputs
print("Are the outputs identical?", manual_chatml.strip() == template_chatml.strip())
print()
if manual_chatml.strip() != template_chatml.strip():
    # Show where they differ
    lines_manual   = manual_chatml.strip().splitlines()
    lines_template = template_chatml.strip().splitlines()
    max_lines = max(len(lines_manual), len(lines_template))
    for i in range(max_lines):
        lm = lines_manual[i]   if i < len(lines_manual)   else "<missing>"
        lt = lines_template[i] if i < len(lines_template) else "<missing>"
        marker = "==" if lm == lt else "!!"
        print(f"  [{marker}] line {i:2d}  manual={lm!r}  template={lt!r}")
else:
    print("Both methods produce the same ChatML string — good!")

Are the outputs identical? True

Both methods produce the same ChatML string — good!


## Part 3: Dataset Shapes

Fine-tuning frameworks accept data in several formats. Knowing these shapes lets you convert between them.

### Shape 1: `messages` format
A list of `{role, content}` dicts per example. Used directly by TRL's `SFTTrainer` with `dataset_text_field` pointing to a pre-formatted string, or via TRL's chat-template collator.

### Shape 2: `prompt / completion` format
Two separate string fields. Common in OpenAI fine-tuning API, older HuggingFace examples.

### Shape 3: Packed sequences
Multiple short examples concatenated with EOS tokens into a single fixed-length sequence. Maximizes GPU utilization by avoiding wasted padding tokens.

In [8]:
# Shape 1: messages format
messages_format = [
    {
        "messages": [
            {"role": "system",    "content": "You are a resume assistant."},
            {"role": "user",      "content": "What programming languages do you know?"},
            {"role": "assistant", "content": "I am proficient in Python, TypeScript, and Go."},
        ]
    },
    {
        "messages": [
            {"role": "system",    "content": "You are a resume assistant."},
            {"role": "user",      "content": "Describe your most recent role."},
            {"role": "assistant", "content": "I was a Senior ML Engineer at Acme Corp, leading a team of 4."},
        ]
    },
]

print("Shape 1: messages format")
for i, ex in enumerate(messages_format):
    print(f"  Example {i}: {len(ex['messages'])} turns")
print()

Shape 1: messages format
  Example 0: 3 turns
  Example 1: 3 turns



In [9]:
# Shape 2: prompt / completion format
prompt_completion_format = [
    {
        "prompt":     "<|im_start|>user\nWhat programming languages do you know?<|im_end|>\n<|im_start|>assistant\n",
        "completion": "I am proficient in Python, TypeScript, and Go.<|im_end|>"
    },
    {
        "prompt":     "<|im_start|>user\nDescribe your most recent role.<|im_end|>\n<|im_start|>assistant\n",
        "completion": "I was a Senior ML Engineer at Acme Corp, leading a team of 4.<|im_end|>"
    },
]

print("Shape 2: prompt / completion format")
for i, ex in enumerate(prompt_completion_format):
    print(f"  Example {i}: prompt={len(ex['prompt'])} chars, completion={len(ex['completion'])} chars")
print()

Shape 2: prompt / completion format
  Example 0: prompt=89 chars, completion=56 chars
  Example 1: prompt=81 chars, completion=71 chars



In [10]:
# Shape 3: packed sequences — show how show_dataset_stats works
from src.data_prep import pack_sequences

# First, convert our messages to flat text records
flat_records = []
for ex in messages_format:
    text = format_chatml(ex["messages"])
    flat_records.append({"text": text})

# Add a few more short records to demonstrate packing
for i in range(5):
    flat_records.append({"text": f"<|im_start|>user\nShort question {i}.<|im_end|>\n<|im_start|>assistant\nShort answer {i}.<|im_end|>"})

print("Before packing:")
stats_before = show_dataset_stats(flat_records)
print()

packed_records = pack_sequences(flat_records, max_len=300)

print()
print("After packing (max_len=300 chars):")
stats_after = show_dataset_stats(packed_records)

Before packing:
[data_prep] Dataset stats:
  count       : 7
  avg chars   : 124.7
  min chars   : 92
  max chars   : 210
  sample entry: {"text": "<|im_start|>system\nYou are a resume assistant.<|im_end|>\n<|im_start|>user\nWhat programming languages do you know?<|im_end|>\n<|im_start|>assistant\nI am proficient in Python, TypeScript, 

[data_prep] Packed 7 records into 5 sequences (max_len=300 chars, text_field='text')

After packing (max_len=300 chars):
[data_prep] Dataset stats:
  count       : 5
  avg chars   : 192.8
  min chars   : 105
  max chars   : 223
  sample entry: {"text": "<|im_start|>system\nYou are a resume assistant.<|im_end|>\n<|im_start|>user\nWhat programming languages do you know?<|im_end|>\n<|im_start|>assistant\nI am proficient in Python, TypeScript, 


### TODO 2

In the cell below, create **3 example training records in `messages` format** for a resume Q&A task. Think like a recruiter: what questions would they actually ask a candidate?

Then answer in the markdown cell:
- What 3 questions did you choose? Why would a recruiter ask them?
- What system prompt did you use? How does it shape the expected answer tone?

In [19]:
# TODO 2: Create 3 training records in messages format for a resume Q&A task
my_training_records = [
    # Record 1
    {
        "messages": [
            {"role": "system",    "content": "You are a professional job candidate answering recruiter questions about your resume. Answer in first person, stay concise, and highlight relevant skills, and fit for the role."},
            {"role": "user",      "content": "Can you summarize your background and the type of role you are looking for?"},  # <- CHANGE THIS
            {"role": "assistant", "content": "I have experience in software development and machine learning, with a strong interest in building practical AI systems. I am looking for a role where I can apply Python, data processing, and model development skills to real-world problems."},  # <- CHANGE THIS
        ]
    },
    # Record 2 — add your question
    {
        "messages": [
            {"role": "system",    "content": "You are a professional job candidate answering recruiter questions about your resume. Answer in first person, stay concise, and highlight relevant skills, and fit for the role."},
            {"role": "user",      "content": "What technical project are you most proud of, and what was your contribution?"},  # <- CHANGE THIS
            {"role": "assistant", "content": "I am most proud of building an ML-related project where I handled data preparation, model experimentation, and evaluation. My contribution was turning an open-ended problem into a working pipeline and explaining the results clearly."},  # <- CHANGE THIS
        ]
    },
    # Record 3 — add your question
    {
        "messages": [
            {"role": "system",    "content": "You are a professional job candidate answering recruiter questions about your resume. Answer in first person, stay concise, and highlight relevant skills, and fit for the role."},
            {"role": "user",      "content": "How do you approach learning new tools or technologies for a project?"},  # <- CHANGE THIS
            {"role": "assistant", "content": "I start by understanding the project goal, then focus on the parts of the tool that are most relevant to solving the problem. I usually learn through documentation, small experiments, and applying the tool directly in the project."},  # <- CHANGE THIS
        ]
    },
]

print(f"Created {len(my_training_records)} training records")
for i, rec in enumerate(my_training_records):
    user_q = next(m['content'] for m in rec['messages'] if m['role'] == 'user')
    print(f"  Record {i+1}: {user_q!r}")

Created 3 training records
  Record 1: 'Can you summarize your background and the type of role you are looking for?'
  Record 2: 'What technical project are you most proud of, and what was your contribution?'
  Record 3: 'How do you approach learning new tools or technologies for a project?'


In [20]:
#Actu -- edit your answer below, then run this cell.
todo2_reflection = """
I chose questions about my background, a technical project, and how I learn new tools because these are realistic recruiter questions that evaluate fit, experience, ownership, and adaptability. A recruiter would ask the background question to understand my career direction, the project question to assess technical impact, and the learning question to see how I handle unfamiliar technologies. 
The system prompt tells the assistant to answer as a professional job candidate in first person, which makes the responses concise, resume-focused, and appropriate for a recruiter conversation."""
print(todo2_reflection)



I chose questions about my background, a technical project, and how I learn new tools because these are realistic recruiter questions that evaluate fit, experience, ownership, and adaptability. A recruiter would ask the background question to understand my career direction, the project question to assess technical impact, and the learning question to see how I handle unfamiliar technologies. 
The system prompt tells the assistant to answer as a professional job candidate in first person, which makes the responses concise, resume-focused, and appropriate for a recruiter conversation.


## Summary

In [21]:
tracker.report()

# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="02",
    section_title="Data Formats & Chat Templates",
    reflection_content=section_text,
)
print("Reflection auto-saved to outputs/homework_reflection.md")


API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

Reflection auto-saved to outputs/homework_reflection.md
